# 06 - SHAP Explainability

explain individual risk predictions from the xgboost model with shap.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
import shap

LABEL = "high_cost"
MODEL_NAME = "xgboost"

FEATURE_COLS = [
    "inpatient_admits", "outpatient_visits", "carrier_claims",
    "inpatient_paid", "outpatient_paid", "carrier_paid",
    "has_alzheimers", "has_chf", "has_ckd", "has_cancer", "has_copd",
    "has_depression", "has_diabetes", "has_ihd", "has_osteoporosis",
    "has_ra_oa", "has_stroke_tia", "has_esrd",
    "age", "is_male", "race_black", "race_other", "race_hispanic",
    "rx_fill_count", "rx_total_cost", "rx_patient_paid",
    "other_payer_paid",
]


In [ ]:
# load the trained model and the feature table
model = joblib.load(f"../data/processed/models/{MODEL_NAME}_{LABEL}.joblib")
df = pd.read_parquet("../data/processed/features_labeled.parquet")
X = df[FEATURE_COLS]


In [ ]:
# tree explainer works directly on xgboost's raw trees, no surrogate needed
explainer = shap.TreeExplainer(model)
shap_values = explainer(X)


In [ ]:
# global feature importance: mean absolute shap value per feature
shap.summary_plot(shap_values, X, plot_type="bar", show=True)


In [ ]:
# full summary plot: direction + magnitude of each feature's effect
shap.summary_plot(shap_values, X, show=True)


In [ ]:
# drill into one member's individual risk explanation
member_idx = 0
shap.plots.waterfall(shap_values[member_idx])


In [ ]:
# save per-member shap values for the dashboard drill-down view
per_member = pd.DataFrame(shap_values.values, columns=FEATURE_COLS)
per_member.insert(0, "desynpuf_id", df["desynpuf_id"].values)

out_path = f"../data/processed/{MODEL_NAME}_{LABEL}_shap_values.parquet"
per_member.to_parquet(out_path, index=False)
print(f"wrote {out_path}")
